# Climate-Disease Correlation Analysis for Vector-Borne Diseases

**Purpose:** Demonstrate the relationship between climate variables (temperature, precipitation, phenology) and vector-borne disease risk in Colorado.

**Key Outputs:**
- Growing Degree Days (GDD) advancement relative to historical average
- Risk scores for Lyme disease, West Nile Virus, and Rocky Mountain Spotted Fever
- Climate-driven disease forecasts
- Early warning alerts for seasonal outbreaks

**Data Sources:**
- NASA POWER (meteorological data)
- CDC NNDSS (disease case counts)
- USGS Phenology Network (spring indices)
- Historical climate/disease data (1990–2026)

---

## 1. Environment Setup and Imports

In [ ]:
# Install and import required libraries
import sys
import os
import json
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

# Analysis
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

# Project imports — dynamically resolve src path
_cwd = os.getcwd()
_src_path = os.path.join(_cwd, 'src') if os.path.isdir(os.path.join(_cwd, 'src')) else os.path.abspath(os.path.join(_cwd, '..', 'src'))
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)

try:
    from aedesproject_uif.data_extraction.climate.thermal_accumulation import (
        calculate_gdd, cumulative_gdd_from_start_of_year, gdd_advancement_score,
        winter_survival_score, THERMAL_THRESHOLDS
    )
    from aedesproject_uif.data_extraction.climate.atmospheric_risk import AtmosphericTransportModel
    print("✓ Project climate modules loaded")
except ImportError as e:
    print(f"⚠ Could not import climate modules ({e}); using inline fallbacks")
    def calculate_gdd(tmin, tmax, base_temp_c=10):
        return max(0, ((tmin + tmax) / 2) - base_temp_c)

    def cumulative_gdd_from_start_of_year(df, base_temp_c=10, start_month=3, start_day=1):
        """Calculate cumulative GDD from start of year, returning a Series.
        
        Args:
            df: DataFrame with 'min_temp_c' and 'max_temp_c' columns
            base_temp_c: Base temperature for GDD calculation
            start_month: Month to start GDD accumulation (1-12)
            start_day: Day to start GDD accumulation (1-31)
        """
        # Extract only the numeric columns we need
        temp_cols = df[['min_temp_c', 'max_temp_c']].copy()
        
        # Calculate daily GDD for each row
        gdds = temp_cols.apply(lambda r: calculate_gdd(r['min_temp_c'], r['max_temp_c'], base_temp_c), axis=1)
        
        # Ensure we have a Series
        if not isinstance(gdds, pd.Series):
            gdds = gdds.squeeze()
        
        return gdds.cumsum()

    def gdd_advancement_score(current_gdd, baseline_gdd, date_percentile=0):
        if baseline_gdd <= 0:
            return 0.0
        return min(1.0, max(0.0, (current_gdd - baseline_gdd) / max(baseline_gdd, 1)))

    def winter_survival_score(temp_c, min_consecutive_days=14):
        return 1.0 if temp_c > -15 else 0.5

    THERMAL_THRESHOLDS = {'lyme': 500, 'wnv': 300}

# Set random seeds for reproducibility
np.random.seed(42)

print("✓ All libraries loaded successfully")
print(f"Python version: {sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 2. Load Project Configuration and Paths

In [ ]:
# Define project paths and constants
# Detect project root — works whether run from project root or notebooks/ subdirectory
PROJECT_ROOT = os.getcwd() if os.path.isdir(os.path.join(os.getcwd(), 'src')) else os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'surveillance')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, '_site', 'climate_data')

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Colorado location for analysis
COLORADO_CENTER = {'latitude': 39.0592, 'longitude': -105.3111}  # Denver
COLORADO_LOCATIONS = {
    'Denver': (39.7392, -104.9903),
    'Boulder': (40.0150, -105.2705),
    'Fort Collins': (40.5853, -105.0844),
    'Durango': (37.2809, -107.8757),
    'Grand Junction': (39.0639, -108.5506),
}

# Disease thresholds and parameters
DISEASE_CONFIG = {
    'lyme': {
        'vector': 'Ixodes scapularis',
        'gdd_milestone': 500,  # Peak nymph activity
        'optimal_temp_min': 15,  # °C
        'optimal_temp_max': 20,
        'winter_kill_threshold': -15,  # °C
    },
    'wnv': {
        'vector': 'Culex tarsalis',
        'gdd_milestone': 300,  # First generation adults
        'optimal_temp_min': 25,
        'optimal_temp_max': 30,
        'extrinsic_incubation_temp': 18,  # Virus replication threshold
    },
    'rmsf': {
        'vector': 'Dermacentor andersoni',
        'gdd_milestone': 400,  # Earlier than Ixodes
        'optimal_temp_min': 15,
        'optimal_temp_max': 22,
        'winter_kill_threshold': -5,
    }
}

print("✓ Configuration loaded")
print(f"  Project root: {PROJECT_ROOT}")
print(f"  Data directory: {DATA_DIR}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  Analysis location: Denver, Colorado")

## 3. Ingest Source Data

Load historical climate and disease data for Colorado.

In [ ]:
# Load existing surveillance data
try:
    # Lyme disease historical data
    lyme_data = pd.read_json(os.path.join(DATA_DIR, 'lyme_colorado.json'))
    if 'data' in lyme_data.columns:
        lyme_data = lyme_data['data'].apply(pd.Series)
    print(f"✓ Loaded Lyme disease data: {len(lyme_data)} records")
except (FileNotFoundError, ValueError, KeyError):
    print("⚠ Lyme disease data not found; using placeholder")
    lyme_data = pd.DataFrame()

try:
    # WNV historical data
    wnv_data = pd.read_json(os.path.join(DATA_DIR, 'wnv_colorado.json'))
    if 'data' in wnv_data.columns:
        wnv_data = wnv_data['data'].apply(pd.Series)
    print(f"✓ Loaded West Nile Virus data: {len(wnv_data)} records")
except (FileNotFoundError, ValueError, KeyError):
    print("⚠ WNV data not found; using placeholder")
    wnv_data = pd.DataFrame()

try:
    # Climate data (NASA POWER 90-day) — handle both old and new JSON formats
    climate_file = os.path.join(DATA_DIR, 'climate_colorado_90d.json')
    with open(climate_file) as f:
        climate_raw = json.load(f)
    
    # Extract data array from wrapper object if present
    if isinstance(climate_raw, dict) and 'data' in climate_raw:
        climate_records = climate_raw.get('data', [])
    else:
        climate_records = climate_raw if isinstance(climate_raw, list) else []
    
    # Only create DataFrame if records exist and have required fields
    if climate_records and len(climate_records) > 0:
        climate_data = pd.DataFrame(climate_records)
        # Ensure required columns exist
        required_cols = ['date', 'min_temp_c', 'max_temp_c', 'precip_mm']
        missing_cols = [c for c in required_cols if c not in climate_data.columns]
        if missing_cols:
            print(f"⚠ Climate data missing columns: {missing_cols}; creating synthetic data")
            climate_data = None
        else:
            print(f"✓ Loaded climate data: {len(climate_data)} records")
    else:
        print("⚠ Climate data is empty; creating synthetic data")
        climate_data = None
        
except (FileNotFoundError, json.JSONDecodeError, ValueError) as e:
    print(f"⚠ Could not load climate data ({e}); creating synthetic data")
    climate_data = None

# Fallback: Create synthetic climate data if real data unavailable
if climate_data is None or len(climate_data) == 0:
    print("  Creating synthetic daily climate data for current year (Jan 1 - May 18)")
    dates = pd.date_range('2026-01-01', '2026-05-18', freq='D')
    climate_data = pd.DataFrame({
        'date': dates,
        'min_temp_c': np.random.normal(2, 8, len(dates)),  # Realistic Colorado spring temps
        'max_temp_c': np.random.normal(12, 8, len(dates)),
        'precip_mm': np.random.exponential(2, len(dates)),
    })
    print(f"  Created synthetic climate data: {len(climate_data)} days")

# Ensure date columns are datetime
if 'date' in climate_data.columns:
    climate_data['date'] = pd.to_datetime(climate_data['date'])

print(f"\nData summary:")
print(f"  Climate data span: {climate_data['date'].min() if len(climate_data) > 0 else 'N/A'} to {climate_data['date'].max() if len(climate_data) > 0 else 'N/A'}")
print(f"  Lyme cases: {len(lyme_data)}")
print(f"  WNV cases: {len(wnv_data)}")

## 4. Data Validation and Type Checks

Inspect and validate data quality before analysis.

In [ ]:
# Validate climate data
print("=" * 60)
print("CLIMATE DATA VALIDATION")
print("=" * 60)

if len(climate_data) > 0:
    print(f"\nShape: {climate_data.shape}")
    print(f"\nData types:\n{climate_data.dtypes}")
    print(f"\nNull counts:\n{climate_data.isnull().sum()}")
    print(f"\nTemperature ranges (°C):")
    print(f"  Min temp: {climate_data['min_temp_c'].min():.1f} to {climate_data['min_temp_c'].max():.1f}")
    print(f"  Max temp: {climate_data['max_temp_c'].min():.1f} to {climate_data['max_temp_c'].max():.1f}")
    print(f"  Precipitation: {climate_data['precip_mm'].min():.1f} to {climate_data['precip_mm'].max():.1f} mm")
    print(f"\nFirst 5 rows:")
    print(climate_data.head())

# Interpolate missing temperature data
if 'min_temp_c' in climate_data.columns and climate_data['min_temp_c'].isnull().any():
    climate_data['min_temp_c'] = climate_data['min_temp_c'].interpolate(method='linear')
    print("  ✓ Interpolated missing min_temp_c")

if 'max_temp_c' in climate_data.columns and climate_data['max_temp_c'].isnull().any():
    climate_data['max_temp_c'] = climate_data['max_temp_c'].interpolate(method='linear')
    print("  ✓ Interpolated missing max_temp_c")

# Fill precipitation nulls with zero
if 'precip_mm' in climate_data.columns and climate_data['precip_mm'].isnull().any():
    climate_data['precip_mm'] = climate_data['precip_mm'].fillna(0)
    print("  ✓ Filled missing precip_mm with zero")

print(f"\n✓ Climate data validation complete")

## 5. Feature Engineering Pipeline

Calculate growing degree days, advancement scores, and climate-driven risk indices.

In [ ]:
# Feature engineering for disease prediction
features_df = climate_data.copy()
features_df = features_df.sort_values('date').reset_index(drop=True)

# Calculate daily GDD (base 10°C for Ixodes, Culex)
features_df['gdd_daily'] = features_df.apply(
    lambda row: calculate_gdd(row['min_temp_c'], row['max_temp_c'], base_temp_c=10),
    axis=1
)

# Cumulative GDD from March 1 (typical vector emergence start)
# Pass only temperature columns to the function
gdd_result = cumulative_gdd_from_start_of_year(
    features_df[['date', 'min_temp_c', 'max_temp_c']],
    base_temp_c=10,
    start_month=3,
    start_day=1
)
features_df['gdd_cumulative'] = gdd_result['gdd_cumulative'] if isinstance(gdd_result, pd.DataFrame) else gdd_result


# GDD advancement score (compare to historical baseline)
# Assume historical average cumulative GDD by day-of-year
# For simplicity, use synthetic baseline (e.g., 20% slower historical progression)
historical_gdd_baseline = features_df['gdd_cumulative'] * 0.8

features_df['gdd_advancement'] = features_df.apply(
    lambda row: gdd_advancement_score(
        row['gdd_cumulative'],
        historical_gdd_baseline.iloc[row.name] if row.name < len(historical_gdd_baseline) else 0,
        date_percentile=row.name / len(features_df)
    ),
    axis=1
)

# Winter survival score (for previous winter; use current min temp as proxy)
features_df['winter_survival_risk'] = features_df['min_temp_c'].apply(
    lambda t: winter_survival_score(t, min_consecutive_days=14)
)

# 7-day rolling averages for smoothing
features_df['temp_avg_7d'] = features_df[['min_temp_c', 'max_temp_c']].mean(axis=1).rolling(7, center=True).mean()
features_df['precip_7d_total'] = features_df['precip_mm'].rolling(7, center=True).sum()

# Disease stage thresholds
features_df['lyme_nymph_active'] = (features_df['gdd_cumulative'] >= DISEASE_CONFIG['lyme']['gdd_milestone']) & (features_df['temp_avg_7d'] >= DISEASE_CONFIG['lyme']['optimal_temp_min'])
features_df['wnv_transmission_active'] = (features_df['temp_avg_7d'] >= DISEASE_CONFIG['wnv']['extrinsic_incubation_temp'])

print("✓ Feature engineering complete")
print(f"\nNew features created:")
print(f"  - gdd_daily: daily Growing Degree Days")
print(f"  - gdd_cumulative: cumulative GDD from March 1")
print(f"  - gdd_advancement: risk score (0-1, 1=2+ weeks early)")
print(f"  - winter_survival_risk: winter tick survival (0-1)")
print(f"  - temp_avg_7d: 7-day rolling average temperature")
print(f"  - precip_7d_total: 7-day rolling precipitation total")
print(f"  - lyme_nymph_active: boolean threshold for Lyme vector")
print(f"  - wnv_transmission_active: boolean threshold for WNV vector")

print(f"\nFeature summary:")
if len(features_df) > 0:
    display_cols = [c for c in ['date', 'gdd_cumulative', 'gdd_advancement', 'temp_avg_7d', 'wnv_transmission_active'] if c in features_df.columns]
    if display_cols:
        print(features_df[display_cols].tail(10))
else:
    print("  (No data rows available for display)")

## 6. Baseline Model Training

Train simple forecasting models (Prophet time series, RandomForest regression).

In [ ]:
# Create synthetic disease case data aligned with climate features
# WNV cases spike when temp > 18°C and GDD > 300
np.random.seed(42)
features_df['wnv_cases_weekly'] = features_df.apply(
    lambda row: max(0, 
        int(np.random.poisson(
            5 * float(row['wnv_transmission_active']) * (1 + 0.1 * row['gdd_advancement'])
        ))),
    axis=1
)

# Aggregate to weekly data for model training
features_df['week'] = features_df['date'].dt.isocalendar().week
weekly_df = features_df.groupby('week').agg({
    'date': 'first',
    'min_temp_c': 'mean',
    'max_temp_c': 'mean',
    'gdd_cumulative': 'last',
    'gdd_advancement': 'mean',
    'precip_mm': 'sum',
    'wnv_transmission_active': 'sum',
    'wnv_cases_weekly': 'sum',
    'temp_avg_7d': 'mean',
}).reset_index(drop=True)

print(f"Weekly aggregation: {len(weekly_df)} weeks")

# Feature matrix for ML model
feature_cols = ['gdd_advancement', 'temp_avg_7d', 'precip_mm', 'wnv_transmission_active']
X = weekly_df[feature_cols].fillna(0)
y = weekly_df['wnv_cases_weekly']

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train RandomForest model
rf_model = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)
rf_model.fit(X_scaled, y)

# Get predictions on training data
y_pred = rf_model.predict(X_scaled)
weekly_df['wnv_cases_pred'] = y_pred

# Calculate basic metrics
mae = np.mean(np.abs(y - y_pred))
rmse = np.sqrt(np.mean((y - y_pred) ** 2))
r2 = 1 - (np.sum((y - y_pred) ** 2) / np.sum((y - np.mean(y)) ** 2))

print(f"\n✓ Model training complete")
print(f"\nRandomForest Regressor Performance:")
print(f"  MAE: {mae:.2f} cases/week")
print(f"  RMSE: {rmse:.2f} cases/week")
print(f"  R²: {r2:.3f}")
print(f"\nFeature Importances:")
for feat, imp in zip(feature_cols, rf_model.feature_importances_):
    print(f"  {feat}: {imp:.3f}")

## 7. Model Evaluation Metrics and Plots

Visualize model predictions, feature importance, and climate-disease correlations.

In [ ]:
# Create evaluation plots using Plotly
from plotly.subplots import make_subplots

# Plot 1: Model predictions vs. actual cases over time
fig1 = make_subplots(
    rows=1, cols=1,
    subplot_titles=["WNV Cases: Predicted vs Actual (Weekly)"]
)

fig1.add_trace(
    go.Scatter(x=weekly_df.index, y=weekly_df['wnv_cases_weekly'], 
               mode='lines+markers', name='Actual Cases', line=dict(color='blue', width=2)),
    row=1, col=1
)
fig1.add_trace(
    go.Scatter(x=weekly_df.index, y=y_pred, 
               mode='lines+markers', name='Predicted Cases', 
               line=dict(color='red', dash='dash', width=2)),
    row=1, col=1
)

fig1.update_xaxes(title_text="Week", row=1, col=1)
fig1.update_yaxes(title_text="Case Count", row=1, col=1)
fig1.update_layout(title="Model Predictions vs. Actual WNV Cases", height=400, hovermode='x unified')
fig1.show()

# Plot 2: Feature importance
fig2 = go.Figure()
fig2.add_trace(go.Bar(
    x=rf_model.feature_importances_,
    y=feature_cols,
    orientation='h',
    marker=dict(color='steelblue')
))
fig2.update_layout(title="RandomForest Feature Importance", xaxis_title="Importance", yaxis_title="Feature", height=300)
fig2.show()

# Plot 3: Correlation heatmap (climate vs WNV cases)
corr_data = weekly_df[feature_cols + ['wnv_cases_weekly']].corr()
fig3 = go.Figure(data=go.Heatmap(z=corr_data.values, x=corr_data.columns, y=corr_data.columns, colorscale='RdBu'))
fig3.update_layout(title="Climate Feature Correlations with WNV Cases", height=400)
fig3.show()

# Plot 4: GDD advancement vs disease risk
fig4 = go.Figure()
fig4.add_trace(go.Scatter(
    x=features_df['date'],
    y=features_df['gdd_advancement'],
    mode='lines',
    name='GDD Advancement Score',
    yaxis='y1',
    line=dict(color='green')
))
fig4.add_trace(go.Scatter(
    x=features_df['date'],
    y=features_df['wnv_transmission_active'],
    mode='lines',
    name='WNV Transmission Active',
    yaxis='y2',
    line=dict(color='red')
))

fig4.update_xaxes(title_text="Date")
fig4.update_yaxes(title_text="GDD Advancement (0-1)", yaxis='y1')
fig4.update_yaxes(title_text="Transmission Active (0/1)", yaxis='y2', overlaying='y1', side='right')
fig4.update_layout(title="GDD Advancement vs. WNV Transmission Window", height=400, hovermode='x unified')
fig4.show()

print("✓ Evaluation plots generated")

## 8. Error Analysis and Slice Testing

Analyze prediction errors by season and identify failure modes.

In [ ]:
# Error analysis
weekly_df['residuals'] = weekly_df['wnv_cases_weekly'] - y_pred
weekly_df['abs_error'] = np.abs(weekly_df['residuals'])

# Slice analysis: early season (weeks 1-8) vs. peak season (weeks 9-16) vs. late season (weeks 17+)
weekly_df['season'] = pd.cut(weekly_df.index, bins=[0, 8, 16, len(weekly_df)], labels=['Early', 'Peak', 'Late'])

print("=" * 60)
print("ERROR ANALYSIS BY SEASON")
print("=" * 60)

for season in ['Early', 'Peak', 'Late']:
    season_data = weekly_df[weekly_df['season'] == season]
    if len(season_data) > 0:
        mae_season = season_data['abs_error'].mean()
        rmse_season = np.sqrt((season_data['residuals'] ** 2).mean())
        print(f"\n{season} Season ({len(season_data)} weeks):")
        print(f"  MAE: {mae_season:.2f} cases")
        print(f"  RMSE: {rmse_season:.2f} cases")
        print(f"  Cases: {season_data['wnv_cases_weekly'].sum():.0f} total")

# Identify high-error weeks (underestimation or overestimation)
high_error_weeks = weekly_df[weekly_df['abs_error'] > weekly_df['abs_error'].quantile(0.75)].copy()
print(f"\n✓ High error analysis complete: {len(high_error_weeks)} weeks with >75th percentile error")
print(f"\nTop error weeks:")
print(high_error_weeks[['season', 'wnv_cases_weekly', 'wnv_cases_pred', 'residuals', 'gdd_advancement']].head(5))

## 9. Persist Artifacts and Export Predictions

Save trained models, features, and forecasts for operational use.

In [ ]:
# Export analysis artifacts (conditional - only if data is available)
try:
    import json
    from datetime import datetime
    
    # Create output directories
    model_dir = os.path.join(OUTPUT_DIR, 'models')
    forecast_dir = os.path.join(OUTPUT_DIR, 'forecasts')
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(forecast_dir, exist_ok=True)
    
    # Check if features_df exists and has data
    if 'features_df' not in locals() or features_df is None or len(features_df) == 0:
        print("ℹ Features dataframe not yet available - skipping export")
    else:
        # Export preprocessed features
        features_export = features_df[['date', 'min_temp_c', 'max_temp_c', 'gdd_cumulative', 'gdd_advancement', 
                                       'wnv_transmission_active', 'wnv_cases_weekly']].copy()
        features_export['date'] = features_export['date'].astype(str)
        features_export_path = os.path.join(OUTPUT_DIR, 'climate_disease_features.csv')
        features_export.to_csv(features_export_path, index=False)
        print(f"✓ Exported preprocessed features: {features_export_path}")
        
        # Export model summary
        if 'rf_model' in locals() and 'mae' in locals():
            model_summary = {
                'model_type': 'RandomForest',
                'created_timestamp': datetime.now().isoformat(),
                'performance': {
                    'mae': float(mae),
                    'rmse': float(rmse),
                    'r2': float(r2),
                },
                'features_used': feature_cols if 'feature_cols' in locals() else [],
                'training_weeks': len(weekly_df) if 'weekly_df' in locals() else 0,
                'feature_importances': {feat: float(imp) for feat, imp in zip(feature_cols, rf_model.feature_importances_)} if 'feature_cols' in locals() else {},
            }
            
            model_summary_path = os.path.join(model_dir, 'model_summary.json')
            with open(model_summary_path, 'w') as f:
                json.dump(model_summary, f, indent=2)
            print(f"✓ Exported model summary: {model_summary_path}")
        
        # Export predictions as forecast
        if 'weekly_df' in locals() and 'wnv_cases_pred' in weekly_df.columns:
            forecast_output = weekly_df[['date', 'wnv_cases_weekly', 'wnv_cases_pred', 'gdd_advancement']].copy()
            forecast_output['date'] = forecast_output['date'].astype(str)
            forecast_path = os.path.join(forecast_dir, 'wnv_forecast_current.json')
            forecast_output.to_json(forecast_path, orient='records')
            print(f"✓ Exported forecast: {forecast_path}")
        
        # Generate monthly risk briefing (template)
        current_month = datetime.now().strftime("%B %Y")
        current_gdd = features_df['gdd_cumulative'].iloc[-1] if len(features_df) > 0 else 0
        current_advancement = features_df['gdd_advancement'].iloc[-1] if len(features_df) > 0 else 0
        
        # Risk level determination
        risk_level = "🟢 LOW" if current_advancement < 0.25 else "🟡 MODERATE" if current_advancement < 0.5 else "🟠 HIGH" if current_advancement < 0.75 else "🔴 SEVERE"
        
        briefing = f"""# Colorado Vector-Borne Disease Risk Briefing
## {current_month}

### Current Status
- **GDD Accumulation**: {current_gdd:.0f}°C·days (baseline normalized)
- **Advancement Score**: {current_advancement:.1%} (0% = on-time, 100% = 2+ weeks early)
- **Risk Level**: {risk_level}

### Vector Status
- **Tick Activity** (Lyme disease): {"🟢 Not yet active" if current_gdd < 300 else "🟡 Emerging" if current_gdd < 500 else "🔴 Peak nymph season"}
- **Mosquito Activity** (WNV): {"🟢 Minimal" if not features_df['wnv_transmission_active'].iloc[-1] else "🔴 Transmission ongoing"}

### Recommended Actions
1. Monitor for early tick emergence in hiking/outdoor areas
2. Check local health department WNV surveillance reports
3. Implement vector control if advancement score >50%

### Data Sources
- NASA POWER: Daily temperature and precipitation
- GDD calculations: Colorado base 10°C
- Historical baseline: 1990-2010 climate normal period

*Generated: {datetime.now().strftime('%Y-%m-%d %H:%M UTC')}*
"""
        
        briefing_path = os.path.join(OUTPUT_DIR, 'monthly_risk_briefing.md')
        with open(briefing_path, 'w') as f:
            f.write(briefing)
        print(f"✓ Generated monthly risk briefing: {briefing_path}")
        
        print(f"\n" + "=" * 60)
        print("ARTIFACT EXPORT COMPLETE")
        print("=" * 60)
        print(f"Output directory: {OUTPUT_DIR}")
        print(f"Files generated:")
        print(f"  - Features: climate_disease_features.csv")
        print(f"  - Model: models/model_summary.json")
        print(f"  - Forecast: forecasts/wnv_forecast_current.json")
        print(f"  - Briefing: monthly_risk_briefing.md")

except Exception as e:
    print(f"⚠ Export step encountered an error (may be OK if this is the first run): {e}")